# Computational Thinking - Practice 01: FastAPI Sentiment Analysis API

**Student:** Nguyễn Tấn Tài (MSSV: 25127493 - Lớp: 25C09)  
**Model:** `distilbert/distilbert-base-uncased-finetuned-sst-2-english`

This notebook sets up and tests the FastAPI web service for English sentiment classification on Google Colab or Jupyter Notebook.

## 1. Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn transformers torch omegaconf pydantic requests nest-asyncio

## 2. Create `config.yaml`

In [ ]:
%%writefile config.yaml
model_path: "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
max_length: 512

## 3. Define `SentimentClassifier` Wrapper

In [ ]:
from typing import Any
from omegaconf import OmegaConf
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification


class SentimentClassifier:
    def __init__(self, config_path: str = "./config.yaml") -> None:
        self.config = OmegaConf.load(config_path)
        self.model_path: str = str(self.config.model_path)
        self.max_length: int = int(self.config.get("max_length", 512))
        self.tokenizer = DistilBertTokenizer.from_pretrained(self.model_path)
        self.model = DistilBertForSequenceClassification.from_pretrained(self.model_path)
        self.model.eval()

    def predict(self, text: str) -> dict[str, Any]:
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
        )
        with torch.no_grad():
            logits = self.model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).squeeze(0)

        predicted_class_id = int(probs.argmax().item())
        label = str(self.model.config.id2label[predicted_class_id])
        confidence = round(float(probs[predicted_class_id].item()), 4)
        scores = {
            str(self.model.config.id2label[idx]): round(float(prob.item()), 4)
            for idx, prob in enumerate(probs)
        }
        return {
            "text": text,
            "label": label,
            "confidence": confidence,
            "scores": scores,
        }


classifier = SentimentClassifier("./config.yaml")
print(classifier.predict("Hello, my dog is cute!"))

## 4. Create FastAPI App & Start Background Server

In [ ]:
import threading
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, field_validator
import nest_asyncio
import uvicorn

nest_asyncio.apply()

app = FastAPI(title="Sentiment Analysis API", version="1.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class PredictRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)

    @field_validator("text")
    @classmethod
    def validate_non_empty_text(cls, value: str) -> str:
        stripped = value.strip()
        if not stripped:
            raise ValueError("Input text cannot be empty or whitespace only.")
        return stripped


@app.get("/")
def root():
    return {
        "status": "ok",
        "api_name": "Hugging Face Sentiment Analysis API",
        "model": classifier.model_path,
        "endpoints": {
            "GET /": "Introduction and API endpoint overview",
            "GET /health": "Operational health check",
            "POST /predict": "Analyze sentiment for an input text payload",
        },
    }


@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "model_loaded": True,
        "model_name": classifier.model_path,
    }


@app.post("/predict")
def predict_sentiment(request: PredictRequest):
    result = classifier.predict(request.text)
    return {
        "status": "success",
        "model": classifier.model_path,
        "input_text": request.text,
        "prediction": {
            "label": result["label"],
            "confidence": result["confidence"],
            "scores": result["scores"],
        },
    }


server_thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "warning"},
    daemon=True,
)
server_thread.start()
time.sleep(2)
print("FastAPI server is running on http://127.0.0.1:8000")

## 5. Test API Endpoints with `requests`

In [ ]:
import json
import requests

BASE_URL = "http://127.0.0.1:8000"

print("--- GET / ---")
print(json.dumps(requests.get(f"{BASE_URL}/").json(), indent=2))

print("\n--- GET /health ---")
print(json.dumps(requests.get(f"{BASE_URL}/health").json(), indent=2))

print("\n--- POST /predict (Positive) ---")
r1 = requests.post(f"{BASE_URL}/predict", json={"text": "I really enjoyed the class today!"})
print(json.dumps(r1.json(), indent=2))

print("\n--- POST /predict (Negative) ---")
r2 = requests.post(f"{BASE_URL}/predict", json={"text": "The movie was boring and a waste of time."})
print(json.dumps(r2.json(), indent=2))

## 6. Optional: Expose Local Server Publicly via Pinggy Tunnel

In [ ]:
# Uncomment the line below on Google Colab if you want a public URL via Pinggy:
# !ssh -o StrictHostKeyChecking=no -p 443 -R0:localhost:8000 qr@a.pinggy.io